In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
# from sklearn.metrics import r2_score
import warnings
from src.features import engineer_features

warnings.filterwarnings('ignore')

In [2]:
print("Loading competition data...")
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train['geohash'] = train['geohash'].fillna('Missing')
test['geohash'] = test['geohash'].fillna('Missing')

test['demand'] = -1 
df = pd.concat([train, test], ignore_index=True)

Loading competition data...


In [3]:
print("Applying advanced spatial-temporal feature engineering...")
df_engineered = engineer_features(df)

train_df = df_engineered[df_engineered['demand'] != -1].copy()
test_df = df_engineered[df_engineered['demand'] == -1].copy()

X = train_df.drop(columns=['Index', 'demand'])
y = train_df['demand']
X_test = test_df.drop(columns=['Index', 'demand'])

cat_features = ['geohash', 'RoadType', 'Weather']

Applying advanced spatial-temporal feature engineering...


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

print("Initiating LightGBM & CatBoost Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # --- LightGBM ---
    X_train_lgb, X_val_lgb, X_test_lgb = X_train.copy(), X_val.copy(), X_test.copy()
    for col in cat_features:
        X_train_lgb[col] = X_train_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    lgb = LGBMRegressor(n_estimators=1500, learning_rate=0.03, random_state=42, n_jobs=-1)
    lgb.fit(
        X_train_lgb, y_train, eval_set=[(X_val_lgb, y_val)], 
        callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=0)]
    )
    lgb_test_preds += lgb.predict(X_test_lgb) / kf.n_splits
    
    # --- CatBoost ---
    cat = CatBoostRegressor(iterations=1500, learning_rate=0.04, cat_features=cat_features, random_seed=42, verbose=0)
    cat.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)
    cat_test_preds += cat.predict(X_test) / kf.n_splits

# Generate Final Blended Predictions
final_predictions = (lgb_test_preds * 0.5) + (cat_test_preds * 0.5)

submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully! 100% Legal submission ready.")

Initiating LightGBM & CatBoost Ensemble...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010735 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1672
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 13
[LightGBM] [Info] Start training from score 0.093784
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1500]	valid_0's l2: 0.000802742
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_featu